In [0]:
%run ../setup_nautiq_dev

# Nautiq - setup del entorno

Configuración centralizada para el flujo activo del TFM:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado técnico de Auto Loader.
- Tablas Silver DEV.
- Gold único para analítica JIT y ML.
- Nombre reservado para el notebook ML de clasificación.
- Cambio futuro entre tablas administradas y ADLS externo.

### Notebooks activos

- `silver_ais_positions_dev`
- `silver_ais_static_dev`
- `gold_vessel_jit_features`
- `ml_vessel_jit_classification` *(se implementará después)*


DataFrame[]

NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 16 elementos encontrados
[OK] ais_static: 24 elementos encontrados

Silver DEV tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold table:
  - masterxyz002dbr.gold.vessel_jit_features

Active notebooks:
  - silver_ais_positions_dev
  - silver_ais_static_dev
  - gold_vessel_jit_features
  - ml_vessel_jit_classification  [future]

[OK] Setup completado correctamente.


In [0]:
# Crea la tabla Delta Silver con columnas normalizadas.

location_clause = "" if positions_target_path is None else f"LOCATION '{positions_target_path}'"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {positions_target_table} (
    mmsi BIGINT COMMENT 'Maritime Mobile Service Identity; partition key en Kafka de origen',
    event_timestamp TIMESTAMP COMMENT 'Instante del reporte AIS en UTC, derivado del campo timestamp del contrato',
    event_date DATE COMMENT 'Fecha UTC derivada del instante del reporte AIS',
    latitude DOUBLE COMMENT 'Latitud [-90, 90]',
    longitude DOUBLE COMMENT 'Longitud [-180, 180]',
    speed_over_ground_knots DOUBLE COMMENT 'Speed Over Ground en nudos (>= 0)',
    course_over_ground_degrees DOUBLE COMMENT 'Course Over Ground en grados',
    true_heading_degrees INT COMMENT 'True heading en grados',
    navigation_status_code INT COMMENT 'Navigational status AIS',
    correlation_id STRING COMMENT 'Identificador de correlacion y trazabilidad del evento',
    bronze_ingested_timestamp TIMESTAMP COMMENT 'Instante de ingestion informado en _ingested_at del Parquet Bronze',
    kafka_ingestion_timestamp TIMESTAMP COMMENT 'Instante de ingestion informado en el Parquet Bronze por el pipeline Kafka/Flink',
    kafka_partition BIGINT COMMENT 'Particion Kafka de origen',
    kafka_offset BIGINT COMMENT 'Offset Kafka de origen dentro de la particion',
    flink_processing_timestamp TIMESTAMP COMMENT 'Instante de procesamiento informado por Flink',
    schema_version BIGINT COMMENT 'Version del esquema informada en el Parquet Bronze',
    bronze_partition_date DATE COMMENT 'Fecha extraida de la particion _dt del fichero Bronze',
    source_file STRING COMMENT 'Ruta del fichero Parquet de origen en Bronze',
    silver_processed_timestamp TIMESTAMP COMMENT 'Instante en que Databricks proceso el registro en Silver'
)
USING DELTA
COMMENT 'Historico normalizado de reportes AIS de posicion de los buques'
{location_clause}
""")

DataFrame[]

In [0]:
# Lee solamente los nuevos Parquet compactados del topic positions.

positions_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", positions_schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("pathGlobFilter", "compacted-part-*")
    .load(positions_source)
    .selectExpr("*", "_metadata.file_path AS source_file")
)

positions_raw.createOrReplaceTempView("vw_bronze_positions_stream")

In [0]:
# Normaliza nombres y tipos, filtra registros no validos y elimina duplicados Kafka.

positions_silver = (
    spark.sql(f"""
        SELECT
            CAST(mmsi AS BIGINT) AS mmsi,
            CAST(`timestamp` AS TIMESTAMP) AS event_timestamp,
            CAST(CAST(`timestamp` AS TIMESTAMP) AS DATE) AS event_date,
            CAST(lat AS DOUBLE) AS latitude,
            CAST(lon AS DOUBLE) AS longitude,
            CAST(speed AS DOUBLE) AS speed_over_ground_knots,
            CAST(cog AS DOUBLE) AS course_over_ground_degrees,
            CAST(heading AS INT) AS true_heading_degrees,
            CAST(nav_status AS INT) AS navigation_status_code,
            CAST(correlation_id AS STRING) AS correlation_id,
            TRY_TO_TIMESTAMP(_ingested_at) AS bronze_ingested_timestamp,
            CAST(_kafka_ingestion_time AS TIMESTAMP) AS kafka_ingestion_timestamp,
            CAST(_kafka_partition AS BIGINT) AS kafka_partition,
            CAST(_kafka_offset AS BIGINT) AS kafka_offset,
            CAST(_flink_processing_time AS TIMESTAMP) AS flink_processing_timestamp,
            CAST(_schema_version AS BIGINT) AS schema_version,
            CAST(REGEXP_EXTRACT(source_file, '_dt=([0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}})', 1) AS DATE) AS bronze_partition_date,
            source_file,
            CURRENT_TIMESTAMP() AS silver_processed_timestamp
        FROM vw_bronze_positions_stream
        WHERE CAST(REGEXP_EXTRACT(source_file, '_dt=([0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}})', 1) AS DATE) >= DATE '{bronze_start_date}'
          AND mmsi IS NOT NULL
          AND correlation_id IS NOT NULL
          AND _kafka_partition IS NOT NULL
          AND _kafka_offset IS NOT NULL
          AND CAST(`timestamp` AS TIMESTAMP) IS NOT NULL
          AND lat BETWEEN -90 AND 90
          AND lon BETWEEN -180 AND 180
          AND (speed IS NULL OR speed >= 0)
    """)
    .dropDuplicates(["kafka_partition", "kafka_offset"])
)

In [0]:
# Escribe los registros validos en Silver.

query = (
    positions_silver.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", positions_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(positions_target_table)
)

query.awaitTermination()

print(f"[OK] Tabla actualizada: {positions_target_table}")

[OK] Tabla actualizada: masterxyz002dbr.silver.ais_positions
